# Step by Step Test Suite for Testing Replay of Matching Process

## Imports

In [ ]:
%cd ..

In [ ]:
import random
import pandas as pd

import wannadb.resources as resources
from wannadb.configuration import BasePipelineElement, Pipeline
from wannadb.data.data import DocumentBase, Document, Attribute
from wannadb.event_logger import BaseEventLogger, MatchingEventLogger
from experiments.automatic_feedback import AutomaticRandomFaultyRankingBasedMatchingFeedback, AutomaticFaultyCustomMatchesRandomRankingBasedMatchingFeedback
from wannadb.interaction import EmptyInteractionCallback
from wannadb.matching.custom_match_extraction import FaissSentenceSimilarityExtractor
from wannadb.matching.distance import SignalsMeanDistance
from wannadb.matching.matching import RankingBasedMatcher
from wannadb.matching.matching_replay import RankingBasedMatchingReplayer
from wannadb.preprocessing.embedding import BERTContextSentenceEmbedder,RelativePositionEmbedder, SBERTDocumentSentenceEmbedder, SBERTLabelEmbedder, SBERTTextEmbedder
from wannadb.preprocessing.extraction import SpacyNERExtractor, StanzaNERExtractor
from wannadb.preprocessing.label_paraphrasing import OntoNotesLabelParaphraser, SplitAttributeNameLabelParaphraser
from wannadb.preprocessing.normalization import CopyNormalizer
from wannadb.preprocessing.other_processing import ContextSentenceCacher
from wannadb.statistics import Statistics
from wannadb.status import EmptyStatusCallback

## Util

In [ ]:
def setup_dataset(dataset) -> tuple[Statistics, DocumentBase]:
    """
    sets up a given dataset and returns Statistics and ASETDocumentBase
    """
    documents = dataset.load_dataset()

    user_attribute_names = dataset.ATTRIBUTES

    # create document base
    document_base = DocumentBase(
        documents=[Document(doc["id"], doc["text"]) for doc in documents],
        attributes=[Attribute(attribute_name) for attribute_name in user_attribute_names]
    )

    statistics = Statistics(do_collect=True)

    statistics["user_provided_attribute_names"] = user_attribute_names
    statistics["dataset"]["dataset_name"] = dataset.NAME
    statistics["dataset"]["attributes"] = dataset.ATTRIBUTES
    statistics["dataset"]["num_documents"] = len(documents)
    statistics["preprocessing"]["num_nuggets"] = 0
    statistics["preprocessing"]["num_has_examples"] = 0

    return statistics, document_base

def run_preprocessing_pipeline(statistics : Statistics, document_base : DocumentBase) -> DocumentBase:
    """
    Runs given combination of pipeline elements and stores them into return dictionary
    """
    
    # create pipeline
    preprocessing_phase = Pipeline([
        StanzaNERExtractor(),
        SpacyNERExtractor("SpacyEnCoreWebLg"),
        ContextSentenceCacher(),
        CopyNormalizer(),
        OntoNotesLabelParaphraser(),
        SplitAttributeNameLabelParaphraser(do_lowercase=True, splitters=[" ", "_"]),
        SBERTLabelEmbedder("SBERTBertLargeNliMeanTokensResource"),
        SBERTTextEmbedder("SBERTBertLargeNliMeanTokensResource"),
        BERTContextSentenceEmbedder("BertLargeCasedResource"),
        SBERTDocumentSentenceEmbedder("SBERTBertLargeNliMeanTokensResource"),
        RelativePositionEmbedder()
    ])

    statistics["preprocessing"]["config"] = preprocessing_phase.to_config()

    # call pipeline
    preprocessing_phase(
        document_base=document_base,
        interaction_callback=EmptyInteractionCallback(),
        status_callback=EmptyStatusCallback(),
        statistics=statistics["preprocessing"]
    )

    return document_base

def run_matching_pipeline(statistics : Statistics, document_base : DocumentBase, dataset) -> DocumentBase:
    """
    Runs given combination of pipeline elements and stores them into return dictionary
    """
    documents = dataset.load_dataset()
    
    matching_phase = Pipeline(
                [
                    SplitAttributeNameLabelParaphraser(do_lowercase=True, splitters=[" ", "_"]),
                    ContextSentenceCacher(),
                    SBERTLabelEmbedder("SBERTBertLargeNliMeanTokensResource"),
                    SBERTDocumentSentenceEmbedder("SBERTBertLargeNliMeanTokensResource"),
                    RankingBasedMatcher(
                        distance=SignalsMeanDistance(
                            signal_identifiers=[
                                "LabelEmbeddingSignal",
                                "TextEmbeddingSignal",
                                "ContextSentenceEmbeddingSignal",
                                "RelativePositionSignal"
                            ]
                        ),
                        max_num_feedback=100,
                        len_ranked_list=10,
                        max_distance=0.2,
                        num_random_docs=1,
                        sampling_mode="AT_MAX_DISTANCE_THRESHOLD",
                        adjust_threshold=True,
                        nugget_pipeline=Pipeline(
                            [
                                ContextSentenceCacher(),
                                CopyNormalizer(),
                                OntoNotesLabelParaphraser(),
                                SplitAttributeNameLabelParaphraser(do_lowercase=True, splitters=[" ", "_"]),
                                SBERTLabelEmbedder("SBERTBertLargeNliMeanTokensResource"),
                                SBERTTextEmbedder("SBERTBertLargeNliMeanTokensResource"),
                                BERTContextSentenceEmbedder("BertLargeCasedResource"),
                                RelativePositionEmbedder()
                            ]
                        ),
                        find_additional_nuggets=FaissSentenceSimilarityExtractor(num_similar_sentences=20, num_phrases_per_sentence=3),
                        store_best_guesses=True,
                        event_logger=MatchingEventLogger()
                    )
                ]
            )

    statistics["matching"]["config"] = matching_phase.to_config()
    
    feedback = AutomaticFaultyCustomMatchesRandomRankingBasedMatchingFeedback(
        documents,
        {attr_name: attr_name for attr_name in dataset.ATTRIBUTES},
        faulty_probability=0.2
    )
    matching_phase(
        document_base=document_base,
        interaction_callback=feedback,
        status_callback=EmptyStatusCallback(),
        statistics=statistics["matching"]
    )
    
    return document_base, feedback._faulty_feedback_list

## Setup

In [ ]:
from experiments.datasets.aviation import aviation

dataset = aviation
documents = dataset.load_dataset()

if resources.MANAGER is None:
    resource_manager = resources.ResourceManager()

## Run Pipeline

In [ ]:
random.seed(42) # set seed for reproducibility of random feedback in the matching phase
if False: # set to True to run preprocessing, otherwise it will just load the preprocessed data from cache
    statistics, data_base = setup_dataset(dataset)
    data_base = run_preprocessing_pipeline(statistics, data_base)
    byte_data = data_base.to_bson()
    with open("cache/data_base_with_custom_matches.bson", "wb") as f:
        f.write(byte_data)
else:
    with open("cache/data_base_with_custom_matches.bson", "rb") as f:
        byte_data = f.read()
    data_base = DocumentBase.from_bson(byte_data)
    data_base.documents.sort(key=lambda doc: doc.name) # sort documents by name to ensure same order as in replay history, which is important for the replay to work correctly, as it relies on the order of documents and interactions in the history to match them with the current document base during replay. If the order is different, it can lead to mismatches and incorrect replay behavior.
    data_base.attributes.sort(key=lambda attr: attr.name) # sort attributes by name to ensure same order as in replay history, which is important for the replay to work correctly, as it relies on the order of documents and interactions in the history to match them with the current document base during replay. If the order is different, it can lead to mismatches and incorrect replay behavior.
    statistics = Statistics(do_collect=True)

In [ ]:
random.seed(42) # set seed for reproducibility of random feedback in the matching phase
data_base, faulty_feedback_list = run_matching_pipeline(statistics, data_base, dataset)
byte_data = data_base.to_bson()
with open("cache/matched_with_custom_matches.bson", "wb") as f:
    f.write(byte_data)
byte_data = data_base.to_bson(save_attribute_mappings=False)
with open("cache/no_matching_with_custom_nuggets.bson", "wb") as f:
    f.write(byte_data)

## Replay

In [ ]:
replayer = RankingBasedMatchingReplayer(
    distance=SignalsMeanDistance(
        signal_identifiers=[
            "LabelEmbeddingSignal",
            "TextEmbeddingSignal",
            "ContextSentenceEmbeddingSignal",
            "RelativePositionSignal"
        ]
    ),
    max_num_feedback=100,
    len_ranked_list=10,
    max_distance=0.2,
    num_random_docs=1,
    sampling_mode="AT_MAX_DISTANCE_THRESHOLD",
    adjust_threshold=True,
    nugget_pipeline=Pipeline(
        [
            ContextSentenceCacher(),
            CopyNormalizer(),
            OntoNotesLabelParaphraser(),
            SplitAttributeNameLabelParaphraser(do_lowercase=True, splitters=[" ", "_"]),
            SBERTLabelEmbedder("SBERTBertLargeNliMeanTokensResource"),
            SBERTTextEmbedder("SBERTBertLargeNliMeanTokensResource"),
            BERTContextSentenceEmbedder("BertLargeCasedResource"),
            RelativePositionEmbedder()
        ]
    ),
    find_additional_nuggets=FaissSentenceSimilarityExtractor(num_similar_sentences=20, num_phrases_per_sentence=3),
    store_best_guesses=True
)

In [ ]:
with open("cache/no_matching_with_custom_nuggets.bson", "rb") as f:
    byte_data = f.read()
document_base = DocumentBase.from_bson(byte_data)
document_base.documents.sort(key=lambda doc: doc.name) # sort documents by name to ensure same order as in replay history, which is important for the replay to work correctly, as it relies on the order of documents and interactions in the history to match them with the current document base during replay. If the order is different, it can lead to mismatches and incorrect replay behavior.
document_base.attributes.sort(key=lambda attr: attr.name) # sort attributes by name to ensure same order as in replay history, which is important for the replay to work correctly, as it relies on the order of documents and interactions in the history to match them with the current document base during replay. If the order is different, it can lead to mismatches and incorrect replay behavior.
random.seed(42) # set seed for reproducibility of random feedback in the matching phase
replayer(
    document_base=document_base,
    interaction_callback=EmptyInteractionCallback(),
    status_callback=EmptyStatusCallback(),
    statistics=Statistics(do_collect=True)
)

# Test the Data

In [ ]:
df_org = pd.DataFrame(data_base.to_table_dict())
df_replay = pd.DataFrame(document_base.to_table_dict())
diff = df_org.compare(df_replay, keep_equal=True)

ldt_false = []
for idx, row in diff.iterrows():
    for col in diff.columns.levels[0]:
        self_val = row[(col, "self")]
        other_val = row[(col, "other")]
        if (self_val is None) ^ (other_val is None):
            ldt_false.append((idx, col, self_val, other_val))
            print(f"Difference in row {idx}, column {col}: self={self_val}, other={other_val}")
            continue
        if [v.text for v in self_val] != [v.text for v in other_val]:
            ldt_false.append((idx, col, self_val, other_val))
            print(f"Difference in row {idx}, column {col}: self={self_val[0].text}, other={other_val[0].text}")
assert len(ldt_false) == 0, f"Found differences in {len(ldt_false)} cells between original and replayed document base. See printed output for details."

# Replay with Corrected Faulty Events

In [ ]:
with open("cache/no_matching_with_custom_nuggets.bson", "rb") as f:
    byte_data = f.read()
document_base = DocumentBase.from_bson(byte_data)
document_base.documents.sort(key=lambda doc: doc.name) # sort documents by name to ensure same order as in replay history, which is important for the replay to work correctly, as it relies on the order of documents and interactions in the history to match them with the current document base during replay. If the order is different, it can lead to mismatches and incorrect replay behavior.
document_base.attributes.sort(key=lambda attr: attr.name) # sort attributes by name to ensure same order as in replay history, which is important for the replay to work correctly, as it relies on the order of documents and interactions in the history to match them with the current document base during replay. If the order is different, it can lead to mismatches and incorrect replay behavior.

for event_id, correct_decision in faulty_feedback_list:
    replayer.revert_event(document_base, event_id, correct_decision=correct_decision)